Copyright (c) 2023 Qualcomm Innovation Center, Inc. All rights reserved. <br>
SPDX-License-Identifier: BSD-3-Clause-Clear

**Takeaways:** Users will learn how to onboard a GPT-2 Large network on the Qualcomm® Cloud AI 100 (AIC) Accelerator Card and run inference.

**Before you start:** 
- There are some commands (folder locations etc) that will need to be updated in this notebook based on the platform and installation location. 
- The terms 'model' and 'network' are used interchangeably in this notebook. 

**Last Verified Qualcomm Cloud AI Platform SDK and Apps SDK Version:** Platform SDK 1.12.0.86 and Apps SDK 1.12.0.86

**SKU used:** Cloud AI 100 Standard card

## Introduction 
This notebook is for beginners and will take the user through the workflow, from onboarding the 'GPT-2 Large' model (from HuggingFace) to execution of inference on Cloud AI devices. 

Here is the Cloud AI inference workflow at a high level. 

![Workflow](Images/Workflow.jpg)

We will follow this sequence of steps in the notebook. 

1. **Install required packages**: Begin by installing all the required packages.
2. **Model generation**: Import the model, generate an input.
3. **Compilation**: Compile the model for Qualcomm Cloud AI 100.
4. **Inference on Cloud AI**: Run inference on Cloud AI 100 (AIC) Accelerator Card .

# GPT-2 Large Example

In this example we will download a GPT-2 Large model from HuggingFace and demonstrate the compile and inferencing steps on AIC100.

## 1. Install required packages 

We will install the required packages. 

In [21]:
# Make sure the Python interpreter path is set properly.
import sys
sys.executable

'/local/mnt/workspace/stangade/py3_venv/bin/python3'

In [4]:
!rm -rf gpt2large_venv

!python3.8 -m venv ./gpt2large_venv
!source ./gpt2large_venv/bin/activate
!pip install -r ./requirements.txt

Badly placed ()'s.
  Using cached torch-2.0.1-cp38-cp38-manylinux1_x86_64.whl (619.9 MB)
  Using cached nvidia_cusolver_cu11-11.4.0.1-2-py3-none-manylinux1_x86_64.whl (102.6 MB)
  Using cached nvidia_cusparse_cu11-11.7.4.91-py3-none-manylinux1_x86_64.whl (173.2 MB)
  Using cached nvidia_curand_cu11-10.2.10.91-py3-none-manylinux1_x86_64.whl (54.6 MB)
  Using cached nvidia_cuda_cupti_cu11-11.7.101-py3-none-manylinux1_x86_64.whl (11.8 MB)
  Using cached triton-2.0.0-1-cp38-cp38-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (63.2 MB)
  Using cached nvidia_nvtx_cu11-11.7.91-py3-none-manylinux1_x86_64.whl (98 kB)
  Using cached nvidia_nccl_cu11-2.14.3-py3-none-manylinux1_x86_64.whl (177.1 MB)
  Using cached nvidia_cufft_cu11-10.9.0.58-py3-none-manylinux1_x86_64.whl (168.4 MB)
  Using cached networkx-3.1-py3-none-any.whl (2.1 MB)
     |████████████████████████████████| 153 kB 4.1 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Instal

## 2. Model generation
  * Install patched tranformers library.
  * Generate model. 

### Install patched tranformers library

Cloud AIC100 offers various features to optimize LLM processing. Below modifications to the model are done to achive best performance. The key features which would benefit from model changes are as follows:

1. **Static Input Shapes**: With the static input shapes AIC 100 toolchains can make model work in a efficient way. So, these models need to be modified to support static input shapes.
2. **KV Cache storage on DRAM**: AIC100 supports storing the KV Cache generated by the model on the DRAM. This avoids unnecessary KV cache movements between CPU and AIC100.
3. **Automatic Attention_mask generation**: The model can be modified to generate the attention_mask on AIC100 and store it in the DRAM of AIC100, thus removing the need to move attention_masks between CPU and AIC100 every iteration.

In [6]:
!rm -rf transformers
!git clone --branch v4.33.2 --depth 1 https://github.com/huggingface/transformers

Cloning into 'transformers'...
remote: Enumerating objects: 3928, done.
remote: Counting objects: 100% (3928/3928), done.
remote: Compressing objects: 100% (2966/2966), done.
remote: Total 3928 (delta 1270), reused 2101 (delta 913), pack-reused 0
Receiving objects: 100% (3928/3928), 13.26 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (1270/1270), done.
Note: switching to '6da93f5580e109fad5f7b523cf2b6e8a5bafb623'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [7]:
%cd transformers

/local/mnt/workspace/stangade/llm-models/gpt2-large-single-kv/transformers


In [8]:
!git apply ../0001-Add-KV-Changes-to-GPT2-Models-CausalMask-Fix.patch

In [10]:
!pip install .

Processing /local/mnt/workspace/stangade/llm-models/gpt2-large-single-kv/transformers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for transformers: filename=transformers-4.33.2-py3-none-any.whl size=7637858 sha256=838176225f4974db28e4945c63d854be824786dc29e629864aa76c4f0d3a36fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-2434k0ve/wheels/c0/75/f3/16e2cb0134388ddc34d246770e5aa18d5caa04efe536e8a125
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.33.2
    Uninstalling transformers-4.33.2:
      Successfully uninstalled transformers-4.33.2


In [11]:
%cd ..

/local/mnt/workspace/stangade/llm-models/gpt2-large-single-kv


### Generate model

The generation model script `generateONNX.py` will generate the model in the `<model_base_name>-kv` subdirectory.

In [12]:
!sudo python3 generateONNX.py --model-name gpt2-large --model-class GPT2LMHeadModel --seq-len 128

/local/mnt/workspace/stangade/py3_venv/lib/python3.8/site-packages/transformers/modeling_utils.py:2363: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/local/mnt/workspace/stangade/py3_venv/lib/python3.8/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/local/mnt/workspace/stangade/py3_venv/lib/python3.8/site-packages/transformers/models/gpt2/modeling_gpt2.py:823: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if batch_size <= 0:
/local/mnt/workspace/stangade/py3_venv/lib/python3.8/site-packages/transformers/models/gpt2/modeling_gpt2.py:328: TracerWarning: Converting a tensor to a Pytho

## 3. Compilation

The compile script `compileModel.sh` uses `qaic-exec` cli tool to compile the model for Qualcomm AI Cloud 100. The input to this tool is `onnx` file generated above. The tool produces a QPC (Qualcomm Program Container) binary file in the path defined by `-aic-binary-dir` argument, which is `qpc/`

Below animation shows what happens at the compile time.
<div>
<img src="Images/Network_specialization_gif.gif" width="700"/>
</div>


In [15]:
!rm -rf qpc
!bash compileModel.sh gpt2-large mx6 14

Reading ONNX Model from gpt2-large-kv/generatedModels/gpt2-large-kv_simplified.onnx
Compile started ............... 
Compiling model with FP16 precision.
/transformer/Slice: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/Slice_1: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/h.0/attn/c_attn/Slice: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/h.0/attn/c_proj/Slice: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/h.0/mlp/c_fc/Slice: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/h.0/mlp/c_proj/Slice: end[0] (9223372036854775807) larger than number of elements in this dimension. Clamping to dimension size.
/transformer/h.1/attn/

## 4. Inference on Cloud AI 100 (AIC) Accelerator Card

To perform the inference on Cloud AI 100, we use `runModel.py` where we pass compiled netwrok qpc and setup a session to run the inference on the device. ```qaic``` library is a set of Python APIs that provides support for running inference on AIC100 backend.

Below animation shows what happens at the run time.
<div>
<img src="Images/Prefill_Decode_GIF.gif" width="700"/>
</div>

In [16]:
!sudo python3 runModel.py --model-name gpt2-large --prompt-len 32 --ctx-len 128 --qpc ./qpc/gpt2-large-kv-32pl-128cl-14c/

My name is[' John'][','][' and'][' I']["'m"][' a'][' student'][' at'][' the'][' University'][' of'][' California'][','][' Berkeley']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Republicans']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Democrats']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Republicans']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Democrats']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Republicans']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Democrats']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Republicans']['.'][' I']["'m"][' a'][' member'][' of'][' the'][' Berkeley'][' College'][' Democrats']['.'][' I']["'m"]
E2E: 97.62 tok/s
Loop: 102.42 tok/s


<div style="text-align: right">KBA-220706221635</div>

<div style="text-align: center">Confidential – Qualcomm Technologies, Inc. and/or its affiliated companies – May Contain Trade Secrets</div>

<div style="text-align: center; font-weight: bold">MAY CONTAIN U.S. AND INTERNATIONAL EXPORT CONTROLLED INFORMATION</div>